# 03. PHÂN CHIA DỮ LIỆU, KIỂM SOÁT LEAKAGE & TRÍCH XUẤT REFERENCE SET
---
* **Khung phương pháp luận**: CRISP-DM (Pha 3: Data Preparation & Partitioning)
* **Giao thức thực nghiệm**: Phân chia Stratified Split (80/20), Khóa Test Set, Trích xuất Reference Set cố định (N=500) và Đóng gói Sklearn Pipeline chống rò rỉ
---


---
## 1. THIẾT LẬP MÔI TRƯỜNG & NẠP DỮ LIỆU ĐÃ LÀM SẠCH

Khởi tạo các thư viện tiền xử lý, cấu hình Pandas và nạp tệp `cleaned_data.csv` từ Notebook 02.


In [1]:
# 1.1. Nạp các thư viện tiền xử lý dữ liệu và phân chia tập
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Thiết lập hiển thị số thực
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('[✓] Thư viện và môi trường thực thi đã sẵn sàng!')


[✓] Thư viện và môi trường thực thi đã sẵn sàng!


In [2]:
# 1.2. Định vị và nạp tệp dữ liệu đã làm sạch (cleaned_data.csv)
def locate_cleaned_file():
    candidate_paths = [
        os.path.join('..', 'data', 'processed', 'cleaned_data.csv'),
        os.path.join('data', 'processed', 'cleaned_data.csv'),
        os.path.join('..', '..', 'data', 'processed', 'cleaned_data.csv'),
        os.path.join('credit-risk-xgb-shap', 'data', 'processed', 'cleaned_data.csv')
    ]
    for p in candidate_paths:
        if os.path.exists(p):
            return os.path.abspath(p)
    raise FileNotFoundError('Không tìm thấy tệp cleaned_data.csv!')

cleaned_data_path = locate_cleaned_file()
df = pd.read_csv(cleaned_data_path)
target_col = 'default_payment_next_month'

print(f'[*] Đường dẫn dữ liệu sạch : {cleaned_data_path}')
print(f'[*] Quy mô dữ liệu nạp     : {df.shape[0]:,} dòng x {df.shape[1]} cột')
print(f'[*] Biến mục tiêu          : {target_col} (Tỷ lệ Default: {(df[target_col]==1).mean()*100:.2f}%)')
df.head(3)


[*] Đường dẫn dữ liệu sạch : c:\Users\MSI\Downloads\Specialized Prọect\credit-risk-xgb-shap\data\processed\cleaned_data.csv
[*] Quy mô dữ liệu nạp     : 30,000 dòng x 24 cột
[*] Biến mục tiêu          : default_payment_next_month (Tỷ lệ Default: 22.12%)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_payment_next_month
0,20000.0000,2,2,1,24,2,2,-1,-1,-2,-2,3913.0000,3102.0000,689.0000,0.0000,0.0000,0.0000,0.0000,689.0000,0.0000,0.0000,0.0000,0.0000,1
1,120000.0000,2,2,2,26,-1,2,0,0,0,2,2682.0000,1725.0000,2682.0000,3272.0000,3455.0000,3261.0000,0.0000,1000.0000,1000.0000,1000.0000,0.0000,2000.0000,1
2,90000.0000,2,2,2,34,0,0,0,0,0,0,29239.0000,14027.0000,13559.0000,14331.0000,14948.0000,15549.0000,1518.0000,1500.0000,1000.0000,1000.0000,1000.0000,5000.0000,0


---
## 2. PHÂN CHIA TẬP DỮ LIỆU & GIAO THỨC KIỂM SOÁT LEAKAGE (DATA SPLIT PROTOCOL)

### 2.1. Thiết kế Giao thức Phân chia:
Toàn bộ 30.000 dòng dữ liệu được phân chia thành 2 tập độc lập:
* **Development Set (80% – 24.000 mẫu)**: Môi trường nghiên cứu duy nhất dùng cho trích xuất Reference Set (N=500), 5-Fold Stratified Cross-Validation và tối ưu siêu tham số đa mục tiêu với Optuna.
* **Test Set độc lập (20% – 6.000 mẫu)**: **KHÓA TUYỆT ĐỐI**. Đại diện cho dữ liệu tương lai chưa từng thấy (*Out-of-sample unseen data*), niêm phong toàn bộ để đánh giá nghiệm cuối cùng ở bước kiểm định thống kê DeLong test.

### 2.2. Lấy mẫu Phân tầng (Stratified Sampling):
Sử dụng `stratify=y` với hạt giống ngẫu nhiên cố định `random_state=42` để bảo toàn chính xác tỷ lệ phân phối lớp 22.12% giữa hai tập.


In [3]:
# 2.2. Thực hiện Stratified Train/Test Split (80% Dev vs 20% Test)
X = df.drop(columns=[target_col])
y = df[target_col]

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=RANDOM_SEED, 
    stratify=y
)

# Bảng kiểm toán phân tầng
dev_counts = y_dev.value_counts()
test_counts = y_test.value_counts()
split_summary = pd.DataFrame({
    'Tập dữ liệu': ['Development Set (80%)', 'Test Set Độc lập (20%)', 'Toàn bộ Dữ liệu (100%)'],
    'Tổng số mẫu': [len(y_dev), len(y_test), len(y)],
    'Số khách Tốt (y=0)': [dev_counts[0], test_counts[0], (y==0).sum()],
    'Số khách Vỡ nợ (y=1)': [dev_counts[1], test_counts[1], (y==1).sum()],
    'Tỷ lệ Vỡ nợ (%)': [
        (y_dev == 1).mean() * 100,
        (y_test == 1).mean() * 100,
        (y == 1).mean() * 100
    ]
})

print('=' * 85)
print('BẢNG KIỂM TOÁN TÍNH PHÂN TẦNG CỦA DỮ LIỆU PHÂN CHIA:')
print('=' * 85)
display(split_summary)
print('=' * 85)
print('[✓] Phân chia hoàn tất: Cả Dev Set và Test Set đều duy trì chính xác 22.12% tỷ lệ vỡ nợ!')
print('[✓] Test Set (6.000 dòng) được niêm phong và đóng băng hoàn toàn khỏi quy trình huấn luyện!')


BẢNG KIỂM TOÁN TÍNH PHÂN TẦNG CỦA DỮ LIỆU PHÂN CHIA:


,Tập dữ liệu,Tổng số mẫu,Số khách Tốt (y=0),Số khách Vỡ nợ (y=1),Tỷ lệ Vỡ nợ (%)
0,Development Set (80%),24000,18691,5309,22.1208
1,Test Set Độc lập (20%),6000,4673,1327,22.1167
2,Toàn bộ Dữ liệu (100%),30000,23364,6636,22.1200


[✓] Phân chia hoàn tất: Cả Dev Set và Test Set đều duy trì chính xác 22.12% tỷ lệ vỡ nợ!
[✓] Test Set (6.000 dòng) được niêm phong và đóng băng hoàn toàn khỏi quy trình huấn luyện!


---
## 3. TRÍCH XUẤT VÀ KHÓA CỐ ĐỊNH REFERENCE SET CHO SHAP

### 3.1. Cơ sở Phương pháp luận:
Trong khuôn khổ đề tài, **Độ ổn định giải thích SHAP ($f_2$)** được đo lường bằng hệ số tương quan thứ hạng Spearman trung bình của Global SHAP Importance giữa các lần huấn luyện lặp (Cross-Validation folds).

* **Loại trừ phương sai lấy mẫu**: Nếu mỗi fold tính SHAP trên một tập cá thể khác nhau, biến động thứ hạng SHAP sẽ bị chi phối bởi sai số chọn mẫu chứ không phản ánh bản chất của mô hình $\theta$.
* **Quy trình trích xuất**: Trích xuất phân tầng $N_{ref} = 500$ mẫu từ `Development Set` với hạt giống cố định `random_state=123`.
* **Nguyên tắc chống rò rỉ**: Reference Set **bắt buộc trích từ Development Set, tuyệt đối không trích từ Test Set**.


In [4]:
# 3.2. Trích xuất Reference Set N=500 mẫu phân tầng từ Development Set
REF_SIZE = 500
REF_SEED = 123

_, X_ref, _, y_ref = train_test_split(
    X_dev, y_dev,
    test_size=REF_SIZE,
    random_state=REF_SEED,
    stratify=y_dev
)

ref_counts = y_ref.value_counts()
print('=' * 75)
print('THÔNG TIN REFERENCE SET CỐ ĐỊNH CHO SHAP:')
print('=' * 75)
print(f'[*] Kích thước Reference Set (N_ref) : {len(X_ref)} quan sát x {X_ref.shape[1]} đặc trưng')
print(f'[*] Số lượng khách hàng Tốt (y=0)    : {ref_counts[0]} ({ref_counts[0]/REF_SIZE*100:.2f}%)')
print(f'[*] Số lượng khách hàng Vỡ nợ (y=1) : {ref_counts[1]} ({ref_counts[1]/REF_SIZE*100:.2f}%)')
print(f'[*] Random seed cố định trích xuất   : {REF_SEED}')
print('=' * 75)
print('[✓] Khóa cố định Reference Set thành công! Tập này sẽ được dùng xuyên suốt cho mọi phép tính TreeSHAP.')


THÔNG TIN REFERENCE SET CỐ ĐỊNH CHO SHAP:
[*] Kích thước Reference Set (N_ref) : 500 quan sát x 23 đặc trưng
[*] Số lượng khách hàng Tốt (y=0)    : 389 (77.80%)
[*] Số lượng khách hàng Vỡ nợ (y=1) : 111 (22.20%)
[*] Random seed cố định trích xuất   : 123
[✓] Khóa cố định Reference Set thành công! Tập này sẽ được dùng xuyên suốt cho mọi phép tính TreeSHAP.


---
## 4. ĐÓNG GÓI SCIKIT-LEARN PIPELINE CHỐNG RÒ RỈ THAM SỐ

### 4.1. Nguyên tắc Chuẩn hóa theo Họ Mô hình:
* **Mô hình dạng cây (XGBoost, RF, GBM, LightGBM, CatBoost)**: Bất biến với biến đổi đơn điệu thứ tự ngưỡng ($x_j \le \theta$), không yêu cầu chuẩn hóa thang đo.
* **Mô hình tuyến tính (Logistic Regression)**: Nhạy cảm với biên độ dữ liệu (`LIMIT_BAL` hàng trăm nghìn NT$ vs `EDUCATION` từ 1–4). Bắt buộc chuẩn hóa `StandardScaler`.
* **Kiểm soát rò rỉ**: Bộ chuẩn hóa chỉ được `.fit()` trên Train Fold của từng vòng Cross-Validation, sau đó `.transform()` sang Validation Fold.


In [5]:
# 4.2. Xây dựng ColumnTransformer chuẩn hóa cho Logistic Regression
numeric_features = ['LIMIT_BAL', 'AGE', 
                    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
                    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
other_features = [col for col in X_dev.columns if col not in numeric_features]

preprocessor_scaler = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), numeric_features),
        ('passthrough', 'passthrough', other_features)
    ]
)

# Kiểm thử fit_transform trên Development Set
X_dev_scaled_sample = preprocessor_scaler.fit_transform(X_dev)
print(f'[*] Số đặc trưng chuẩn hóa StandardScaler : {len(numeric_features)}')
print(f'[*] Số đặc trưng giữ nguyên thang đo       : {len(other_features)}')
print(f'[*] Kích thước X_dev sau biến đổi thử nghiệm: {X_dev_scaled_sample.shape}')
print('[✓] Pipeline ColumnTransformer đã sẵn sàng!')


[*] Số đặc trưng chuẩn hóa StandardScaler : 14
[*] Số đặc trưng giữ nguyên thang đo       : 9
[*] Kích thước X_dev sau biến đổi thử nghiệm: (24000, 23)
[✓] Pipeline ColumnTransformer đã sẵn sàng!


---
## 5. XUẤT BẢN CÁC TẬP DỮ LIỆU ĐÃ PHÂN CHIA (DATA ARTIFACTS EXPORT)

Lưu trữ các tệp phân chia vào thư mục `data/processed` để phục vụ cho các notebook tiếp theo:
* `dev.csv` (24.000 dòng): Phục vụ toàn bộ quá trình Cross-Validation và tối ưu siêu tham số.
* `test.csv` (6.000 dòng): **Khóa cố định**, niêm phong cho đánh giá nghiệm cuối cùng.
* `reference_set_500.csv` (500 dòng): Dùng cố định cho giải thích SHAP.


In [6]:
# 5.1. Lưu trữ các tệp phân chia ra thư mục data/processed
def get_target_dir():
    candidates = [
        os.path.abspath(os.path.join('..', 'data', 'processed')),
        os.path.abspath(os.path.join('data', 'processed')),
        os.path.abspath(os.path.join('credit-risk-xgb-shap', 'data', 'processed'))
    ]
    for c in candidates:
        if 'credit-risk-xgb-shap' in c and os.path.exists(os.path.dirname(c)):
            os.makedirs(c, exist_ok=True)
            return c
    os.makedirs(candidates[0], exist_ok=True)
    return candidates[0]

target_dir = get_target_dir()

# 1. Xuất tập Development (X_dev + y_dev)
df_dev_out = X_dev.copy()
df_dev_out[target_col] = y_dev.values
dev_path = os.path.join(target_dir, 'dev.csv')
df_dev_out.to_csv(dev_path, index=False)

# 2. Xuất tập Test độc lập (X_test + y_test)
df_test_out = X_test.copy()
df_test_out[target_col] = y_test.values
test_path = os.path.join(target_dir, 'test.csv')
df_test_out.to_csv(test_path, index=False)

# 3. Xuất Reference Set 500 mẫu cố định (X_ref)
ref_path = os.path.join(target_dir, 'reference_set_500.csv')
X_ref.to_csv(ref_path, index=False)

# Đồng bộ sang thư mục data/processed gốc nếu khác
root_processed = os.path.abspath(os.path.join(os.path.dirname(target_dir), '..', 'data', 'processed'))
try:
    os.makedirs(root_processed, exist_ok=True)
    df_dev_out.to_csv(os.path.join(root_processed, 'dev.csv'), index=False)
    df_test_out.to_csv(os.path.join(root_processed, 'test.csv'), index=False)
    X_ref.to_csv(os.path.join(root_processed, 'reference_set_500.csv'), index=False)
except Exception:
    pass

print('=' * 80)
print('BÁO CÁO XUẤT BẢN ARTIFACTS DỮ LIỆU ĐÃ PHÂN CHIA:')
print('=' * 80)
print(f'1. Development Set : {dev_path} ({df_dev_out.shape[0]:,} dòng x {df_dev_out.shape[1]} cột)')
print(f'2. Test Set (Khóa) : {test_path} ({df_test_out.shape[0]:,} dòng x {df_test_out.shape[1]} cột)')
print(f'3. Reference Set   : {ref_path} ({X_ref.shape[0]:,} dòng x {X_ref.shape[1]} cột)')
print('=' * 80)
print('[✓] Hoàn thành xuất bản! Tất cả tệp dữ liệu đã sẵn sàng cho pha Benchmark và Tối ưu HPO.')


BÁO CÁO XUẤT BẢN ARTIFACTS DỮ LIỆU ĐÃ PHÂN CHIA:
1. Development Set : c:\Users\MSI\Downloads\Specialized Prọect\credit-risk-xgb-shap\data\processed\dev.csv (24,000 dòng x 24 cột)
2. Test Set (Khóa) : c:\Users\MSI\Downloads\Specialized Prọect\credit-risk-xgb-shap\data\processed\test.csv (6,000 dòng x 24 cột)
3. Reference Set   : c:\Users\MSI\Downloads\Specialized Prọect\credit-risk-xgb-shap\data\processed\reference_set_500.csv (500 dòng x 23 cột)
[✓] Hoàn thành xuất bản! Tất cả tệp dữ liệu đã sẵn sàng cho pha Benchmark và Tối ưu HPO.
